In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

from pathlib import Path
import pickle

from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import equinox as eqx
import jax
import jax.numpy as jnp

In [29]:
seq_len = 90
n_nodes = 7000
hidden_size = 32
targets = ['discharge']

key = jax.random.PRNGKey(0)
state = jax.random.uniform(key, (seq_len, n_nodes, hidden_size))
state.shape

(90, 7000, 32)

In [30]:
from jaxtyping import Array, PRNGKeyArray

class GMM(eqx.Module):
    fc1: eqx.nn.Linear
    fc2: eqx.nn.Linear
    _eps: float

    def __init__(self, in_size: int, out_size: int, hidden_size: int, *, key: PRNGKeyArray):
        k1, k2 = jax.random.split(key)
        self.fc1 = eqx.nn.Linear(in_size, hidden_size, key=k1)
        self.fc2 = eqx.nn.Linear(hidden_size, out_size * 3, key=k2)
        self._eps = 1e-5

    def __call__(self, x: Array) -> dict[str, Array]:
        """Perform a GMM head forward pass.

        Parameters
        ----------
        x : Array
            Final latent variables to compute the GMM components.

        Returns
        -------
        dict[str, Array]
            Dictionary containing mixture parameters and weights; where the key 'mu' stores the means, the key
            'sigma' the variances, and the key 'pi' the weights.
        """
        h = jax.nn.relu(self.fc1(x))
        h = self.fc2(h)

        # split output into mu, sigma and weights
        mu, s_latent, p_latent = jnp.split(h, 3, axis=-1)
        sigma = jnp.exp(s_latent) + self._eps
        pi = jax.nn.softmax(p_latent, axis=-1)

        return {'mu': mu, 'sigma': sigma, 'pi': pi}

n_dist_models = 100

heads = {
    target: GMM(hidden_size, n_dist_models, hidden_size * 2, key=key)
    for target in targets
}

heads

{'discharge': GMM(
   fc1=Linear(
     weight=f32[64,32],
     bias=f32[64],
     in_features=32,
     out_features=64,
     use_bias=True
   ),
   fc2=Linear(
     weight=f32[300,64],
     bias=f32[300],
     in_features=64,
     out_features=300,
     use_bias=True
   ),
   _eps=1e-05
 )}

In [31]:
last_state = state[-1,...]
last_state.shape

(7000, 32)

In [43]:
node_mask = jnp.ones((n_nodes,))

(7000,)

In [38]:
#vmap each target over each location
out = {
    target: jax.vmap(head)(last_state)['mu'][...,0]
    for target, head in heads.items()
}

In [41]:
out['discharge'].shape

(7000,)